In [1]:
#I start by opening the smart meter file

import pandas as pd

file_path = 'facility smart meter kw data.xlsx'

df = pd.read_excel(file_path)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# I add the column 'store_status'

df['store_status'] = pd.Series([])
print(df)

                               facility_id  datetime_beginning  kw_demand  \
0     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 16:00:00       3.77   
1     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 17:00:00       3.54   
2     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 18:00:00       4.23   
3     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 19:00:00       3.47   
4     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 20:00:00       3.44   
...                                    ...                 ...        ...   
8370  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 11:00:00       2.53   
8371  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 12:00:00       3.17   
8372  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 13:00:00       2.51   
8373  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 14:00:00       3.14   
8374  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 15:00:00       2.47   

      temp_f  humidity  pressure_millibars store_status  
0      52.65     

In [3]:
#I then fill the new column with 1 when the store is open and 0 when it is closed. By looking at the data, 
#I noticed that the kw_demands spikes regularly from ~4 to ~9 and viceversa. Therefore, I set kw > 9 as the 
#indicator that the shop is open.

df['store_status'] = 0
for n in df['kw_demand']:
    position = df['kw_demand'].tolist().index(n)  
    if n >= 9:
        df.loc[position, 'store_status'] = 1   

print(df)

                               facility_id  datetime_beginning  kw_demand  \
0     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 16:00:00       3.77   
1     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 17:00:00       3.54   
2     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 18:00:00       4.23   
3     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 19:00:00       3.47   
4     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 20:00:00       3.44   
...                                    ...                 ...        ...   
8370  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 11:00:00       2.53   
8371  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 12:00:00       3.17   
8372  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 13:00:00       2.51   
8373  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 14:00:00       3.14   
8374  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 15:00:00       2.47   

      temp_f  humidity  pressure_millibars  store_status  
0      52.65    

In [4]:
# Now I create the ML model. I added the time series features 'year', 'month', 'dayofweek', and 'hour'. Then
# I created the model using the following steps.

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import datetime
import numpy as np

df['year'] = df['datetime_beginning'].dt.year

df['month'] = df['datetime_beginning'].dt.month

df['dayofweek'] = df['datetime_beginning'].dt.dayofweek

df['hour'] = df['datetime_beginning'].dt.hour


# Drop rows with NaN resulting from lagged values
df.dropna(inplace=True)

# Train/Test Split
X = df.drop(['kw_demand', 'datetime_beginning', 'facility_id'], axis=1)  # Features
y = df['kw_demand']  # Target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Selection
model = RandomForestRegressor(random_state=42)

# Training
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("Mean Absolute Error (MAE):", mae)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Percentage Error (MAPE):", mape)

print(df)


Mean Absolute Error (MAE): 0.5308365970149254
Mean Squared Error (MSE): 0.8936154547164179
Root Mean Squared Error (RMSE): 0.9453123582797476
Mean Absolute Percentage Error (MAPE): 10.221090960269667
                               facility_id  datetime_beginning  kw_demand  \
0     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 16:00:00       3.77   
1     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 17:00:00       3.54   
2     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 18:00:00       4.23   
3     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 19:00:00       3.47   
4     93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2023-01-30 20:00:00       3.44   
...                                    ...                 ...        ...   
8370  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 11:00:00       2.53   
8371  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 12:00:00       3.17   
8372  93a64e38-29c9-4e21-a5ca-7ea5fcd61144 2024-01-14 13:00:00       2.51   
8373  93a64e38-29c9-4e21-a5ca-

In [5]:
# Now I open the weather file

file_path = 'facility weather forecast for DR event.xlsx'

df = pd.read_excel(file_path)
print(df)

                              facility_id  datetime_beginning  temp_f  \
0    0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-14 16:00:00   58.66   
1    0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-14 17:00:00   56.52   
2    0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-14 18:00:00   55.35   
3    0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-14 19:00:00   53.87   
4    0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-14 20:00:00   53.21   
..                                    ...                 ...     ...   
139  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-20 11:00:00   56.77   
140  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-20 12:00:00   58.54   
141  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-20 13:00:00   59.58   
142  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-20 14:00:00   60.60   
143  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-20 15:00:00   59.52   

     humidity  pressure_millibars  
0          83              1015.3  
1          86              1015.2  
2          89  

In [6]:
# I filter the weather data set for the days that I will analyze.

start_date = '2024-01-16'
end_date = '2024-01-20'
weather_data_week = df[(df['datetime_beginning'] >= start_date) & (df['datetime_beginning'] <= end_date)]
print(weather_data_week)


                              facility_id  datetime_beginning  temp_f  \
32   0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-16 00:00:00   48.28   
33   0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-16 01:00:00   47.30   
34   0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-16 02:00:00   47.20   
35   0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-16 03:00:00   46.37   
36   0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-16 04:00:00   45.97   
..                                    ...                 ...     ...   
124  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-19 20:00:00   55.96   
125  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-19 21:00:00   54.33   
126  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-19 22:00:00   54.31   
127  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-19 23:00:00   54.52   
128  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-20 00:00:00   55.48   

     humidity  pressure_millibars  
32         87              1017.2  
33         88              1016.3  
34         88  

In [7]:
# Now I create the 'store status' variable for the weather dataset. This time, I do not use kw as parameter to
# identify opening and closing, but rather the time of the day and the day of week. 

weather_data_week['store_status'] = 0

weather_data_week['year'] = weather_data_week['datetime_beginning'].dt.year

weather_data_week['month'] = weather_data_week['datetime_beginning'].dt.month

weather_data_week['dayofweek'] = weather_data_week['datetime_beginning'].dt.dayofweek

weather_data_week['hour'] = weather_data_week['datetime_beginning'].dt.hour


for index, row in weather_data_week.iterrows():
    if (row['hour'] >= 8) and (row['hour'] <= 17) and (row['dayofweek'] > 1):
        weather_data_week.at[index, 'store_status'] = 1


weather_data_week.dropna(inplace=True)


print(weather_data_week)

                              facility_id  datetime_beginning  temp_f  \
32   0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-16 00:00:00   48.28   
33   0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-16 01:00:00   47.30   
34   0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-16 02:00:00   47.20   
35   0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-16 03:00:00   46.37   
36   0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-16 04:00:00   45.97   
..                                    ...                 ...     ...   
124  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-19 20:00:00   55.96   
125  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-19 21:00:00   54.33   
126  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-19 22:00:00   54.31   
127  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-19 23:00:00   54.52   
128  0835fe8e-ea64-400c-8c22-d1f5f8de557c 2024-01-20 00:00:00   55.48   

     humidity  pressure_millibars  store_status  year  month  dayofweek  hour  
32         87              1017.2          

/var/folders/sw/0d21lwk92kq5sh9w8x5brp6c0000gn/T/ipykernel_79937/1730156053.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  weather_data_week['store_status'] = 0
/var/folders/sw/0d21lwk92kq5sh9w8x5brp6c0000gn/T/ipykernel_79937/1730156053.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  weather_data_week['year'] = weather_data_week['datetime_beginning'].dt.year
/var/folders/sw/0d21lwk92kq5sh9w8x5brp6c0000gn/T/ipykernel_79937/1730156053.py:8: SettingWithCopyWarning: 
A value is trying to be set on a co

In [8]:
# Now I use the ML model created before to predict the future kw based on the new 'store_status' variable

X1 = weather_data_week.drop(['datetime_beginning', 'facility_id'], axis=1)
base_case = model.predict(X1)
print(base_case)

[ 3.8155  3.8735  3.8972  3.9004  3.9216  3.947   4.2911  3.1324  3.6735
  7.6038 10.3068 10.5622 10.4038  9.9821 10.2002 10.0529 10.603  10.4313
  3.9119  3.9731  3.9512  3.8747  4.1759  3.7204  3.7963  3.8104  3.8802
  3.8484  3.9422  3.9542  4.1664  3.1328 10.3479 10.6616 10.852  11.3303
 11.5145 11.5264 11.3736 11.6477 11.7514 11.8682  3.8985  3.9818  3.7133
  3.6395  3.7596  3.7617  3.8159  3.8089  3.8664  3.871   3.9085  3.9282
  4.295   3.0966  9.7547 10.5688 10.8393 10.7402 10.8387 10.9739 10.9751
 11.1517 11.4492 11.8791  3.6825  3.8601  4.1872  3.76    3.7051  3.6662
  3.784   3.7871  3.8578  3.8438  3.8141  3.8278  4.1204  3.0476 10.2225
 10.5876 10.7545 11.4391 11.2633 11.4632 11.6046 11.3659 11.4652 11.6628
  3.823   3.9744  4.023   3.9269  3.9282  3.9262  3.9608]


In [9]:
# Now I calculate the earnings from the demand-response event by summing the kw values at the times the store
# would be closed at and multiplying them by 100$/kwh. I also calculate the lost revenue based on the same 
# closing times

earnings = []
for n in [9,10,15,16,17]:
    for m in [0,24,48,72]:
        earnings.append(base_case[n+m]) 
tot_earnings = np.sum(earnings)*100
lost_revenue = len(earnings)*500
print(tot_earnings)
print(lost_revenue)

21750.28
10000


In [10]:
#By joining the demand-response event and remaining closed during the set hours, the store earned $21,750.28 
#from the incentive and lost $10,000 in revenue. Since the net balance is positive, it makes sense to 
#participate in the event.